In [ ]:
from dotenv import load_dotenv
load_dotenv()
import time
import pandas as pd

import os 
import google.generativeai as genai


generation_config = {
    "temperature": 0.8,
    "top_p": 1,
    "top_k": 32,
    "max_output_tokens": 5000
}

safety_settings = [
    {
        "category": "HARM_CATEGORY_HARASSMENT",
        "threshold": "BLOCK_NONE"
    },
    {
        "category": "HARM_CATEGORY_HATE_SPEECH",
        "threshold": "BLOCK_NONE"
    },
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE"
    }
]

In [ ]:
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-pro-latest", 
                              generation_config=generation_config,
                              safety_settings=safety_settings
                              )

In [19]:
import pandas as pd
import numpy as np
import openai
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import re
import logging
import concurrent.futures
from tqdm import tqdm  # Import tqdm for progress bar

# Setup logging
logging.basicConfig(filename='essay_scoring.log', level=logging.INFO, format='%(asctime)s:%(levelname)s:%(message)s')

# Load the dataset
data_path = 'cleandata_t2_no_examples_final.csv'  # Update the path accordingly
dataframe = pd.read_csv(data_path)

# Randomly select 700 rows from the dataset
subset = dataframe.sample(n=400, random_state=42)

# Define the prompt template
prompt_template = """
Please act as an IELTS examiner and assess an essay according to the official scoring criteria. Consider task achievement, coherence and cohesion, lexical resource, and grammatical range and accuracy. Provide a score from 0 to 9 based on the band descriptors.

Band descriptors:
- Band 9 (Expert User): Fully developed response, sophisticated vocabulary and structures, no errors.
- Band 8 (Very Good User): Well-developed, detailed, minor vocabulary and grammar errors.
- Band 7 (Good User): Clear position throughout the response, with relevant ideas. Some errors that do not hinder communication, and good control over complex structures.
- Band 6 (Competent User): Addresses the task with relevant main ideas, though some may lack development. Makes some errors that rarely reduce communication, uses both simple and complex sentence forms.
- Band 5 (Modest User): Addresses the task only partially with limited main ideas. Frequent errors that may cause difficulty for the reader, limited vocabulary range.
- Band 4 (Limited User): Minimal response to the task, unclear position. Frequent errors that may confuse the reader, very limited range of vocabulary and sentence structures.
- Band 3 (Extremely Limited User): Does not adequately address any part of the task. Communication is often distorted by errors.
- Band 2 (Intermittent User): Barely responds to the task, no clear position. Severe control issues with vocabulary and sentence structures.
- Band 1 (Non User): Unrelated to the task, unable to communicate a clear message. Only isolated words correctly formed.
- Band 0: Did not attempt the task.

Please score the essay below and justify your choice based on how well it aligns with the descriptors.

The prompt for the essay:
{prompt_text}

Essay Text:
{essay_text}

Expected response format: Score: [0-9] with justification.
"""

In [ ]:
 # Function to generate scores using gemini
def generate_score(prompt, model="gemini-2.5-pro-latest"):
    try:
        response = model.generate_content(prompt)
        output = response.text.strip()
        
        # Use regex to find the first instance of a numerical score in the output
        match = re.search(r'\b(\d+)\b', output)
        if match:
            score = int(match.group(1))  # Convert the first numeric string found to an integer
            # Ensure the score is within the valid range
            if 0 <= score <= 9:
                return score, output.replace(f"Score: {score}", "").strip()
            else:
                logging.warning(f"Invalid score {score} found. Retrying...")
                return None, None
        else:
            logging.warning("No score found in output. Retrying...")
            return None, None
    except Exception as e:
        logging.error(f"Error generating score: {e}")
        return None, None

# Function to process each essay
def process_essay(index, row, retries=3):
    prompt = prompt_template.format(prompt_text=row['Question'], essay_text=row['Essay'])
    for attempt in range(retries):
        score1, _ = generate_score(prompt)
        score2, _ = generate_score(prompt)
        if score1 is not None and score2 is not None:
            if abs(score1 - score2) <= 2:
                mean_score = (score1 + score2) / 2
                return index, mean_score, f"Scores: {score1}, {score2}"
        logging.warning(f"Scores {score1} and {score2} differed by more than 2. Retrying...")
    # If no valid score after retries, return None
    return index, None, None

# Add new columns for gemini ratings and justifications
subset['gemini_rating'] = np.nan
subset['justification'] = ""

# Use ThreadPoolExecutor for parallel processing
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(process_essay, index, row) for index, row in subset.iterrows()]
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Grading Essays"):
        index, score, justification = future.result()
        subset.at[index, 'gemini_rating'] = score
        subset.at[index, 'justification'] = justification

# Check for NaN values and handle them
if subset['gemini_rating'].isna().any():
    logging.warning("NaN values found in gemini_rating even after retries.")
    # Option: Drop rows with NaN values or handle them as needed
    subset.dropna(subset=['gemini_rating'], inplace=True)

# Convert columns to float if not already
subset['Overall'] = subset['Overall'].astype(float)
subset['gemini_rating'] = subset['gemini_rating'].astype(float)

# Calculate and print the mean squared error to evaluate the model
mse = mean_squared_error(subset['Overall'], subset['gemini_rating'])
print(f"Mean Squared Error: {mse}")
logging.info(f"Mean Squared Error: {mse}")

# Optional: Calculate and print correlation for additional evaluation
correlation = np.corrcoef(subset['Overall'], subset['gemini_rating'])[0, 1]
print(f"Correlation between actual and predicted ratings: {correlation}")
logging.info(f"Correlation between actual and predicted ratings: {correlation}")

# Save the subset with gemini ratings for further analysis
subset.to_csv('output_with_gemini_ratings_v3_4o.csv', index=False)
print("Output saved with gemini ratings v3_40.")
logging.info("Output saved with gemini ratings.")